# Cup with Handle - Active Setup Scanner

**Goal:** find live Cup with Handle setups on the universe RIGHT NOW for charting and watchlist.

**Two scans:**
1. **Recent breakouts** (last 30 days from cached fires) - already-confirmed trades, may still be open
2. **Pending setups** (modified detector) - handle formed in last 30 bars, NOT yet broken out

The deployed v2b filter (stock 50 > 200) is applied to both -- only show setups that pass.

Output is keyed for charting: ticker, all key dates, all key levels, current status.


In [ ]:
# ---------------------------------------------------------------------------
# papermill PARAMETERS (this cell is tagged `parameters`)
#
# papermill injects an override cell immediately BELOW this one, so every path
# derived further down picks the overridden values up. That is why the params
# live in their own cell instead of being tagged onto the config cell -- tagging
# the config cell would leave OUT_DIR and the *_CSV constants derived from the
# ORIGINAL DATA_DIR, silently pointing at Colab paths.
#
# Defaults are the Colab paths, so opening this notebook in Colab and running it
# top-to-bottom behaves exactly as before.
# ---------------------------------------------------------------------------
DATA_DIR = '/content/drive/MyDrive/Bukowski'   # per-ticker <TICKER>.csv corpus
OUT_DIR = ''                                   # '' -> derived as DATA_DIR/results


In [ ]:
# Cell 1 -- Setup
# Colab mount is OPTIONAL: `from google.colab import drive` raises ImportError
# off-Colab (e.g. papermill on the VPS), so the notebook runs in both places
# unchanged. Same guard cup_handle_weekly.ipynb already used.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

import pandas as pd
import numpy as np
from pathlib import Path
import glob
import warnings
warnings.filterwarnings('ignore')

# DATA_DIR / OUT_DIR arrive from the `parameters` cell above. Kept as STRINGS --
# the cells below build paths with f-strings (f'{DATA_DIR}/{ticker}.csv'), so the
# original str type must be preserved.
DATA_DIR = str(DATA_DIR)
OUT_DIR  = str(OUT_DIR) if OUT_DIR else f'{DATA_DIR}/results'

# Tunable: how recent is "active"?
RECENT_BREAKOUT_DAYS    = 30    # for cached fires
PENDING_HANDLE_MAX_BARS = 30    # handle pivot low within last N trading days
PENDING_LOOKBACK_BARS   = 220   # only scan last ~10 months per ticker for speed

# Cup with Handle params (must match the v2b notebook)
ATR_PERIOD = 14
MAX_STOP_PCT   = 0.10
MIN_STOP_ATR   = 1.0
STOCK_FAST_SMA = 50
STOCK_SLOW_SMA = 200

print('Config loaded.')


In [ ]:
# Cell 2 -- Phase 1: Recent breakouts from cached fires
FIRES_CSV = f'{OUT_DIR}/cup_handle_fires.csv'
fires = pd.read_csv(FIRES_CSV, parse_dates=[
    'lch_date','cup_low_date','rch_date','handle_low_date',
    'fire_date','confirm_date','entry_date'
])
print(f'Cached cup_handle fires: {len(fires)}')

today = fires['entry_date'].max()
print(f'Most recent entry in cache: {today.date()}')

recent = fires[fires['entry_date'] >= today - pd.Timedelta(days=RECENT_BREAKOUT_DAYS)].copy()
recent['days_since_entry'] = (today - recent['entry_date']).dt.days
recent = recent.sort_values('entry_date', ascending=False)

print(f'\nBreakouts in last {RECENT_BREAKOUT_DAYS} days: {len(recent)}')
print('\nTop 30 by recency:')
cols = ['ticker','entry_date','days_since_entry','entry_price','stop_price','target_price',
        'cup_depth_pct','handle_retr_pct']
print(recent[cols].head(30).to_string(index=False, float_format=lambda x: f'{x:.2f}'))


In [ ]:
# Cell 3 -- Apply v2b deployed filter: stock 50 > 200 on entry date
# Need to load universe ticker data for SMA lookup
def load_required_tickers(data_dir, ticker_set):
    universe = {}
    for ticker in ticker_set:
        f = f'{data_dir}/{ticker}.csv'
        if not Path(f).exists():
            # try uppercase variants
            for cand in glob.glob(f'{data_dir}/*.csv'):
                if Path(cand).stem.upper() == ticker:
                    f = cand; break
        if not Path(f).exists(): continue
        d = pd.read_csv(f, usecols=['Date','Open','High','Low','Close'])
        d['Date'] = pd.to_datetime(d['Date'])
        d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
        d[f'SMA{STOCK_FAST_SMA}'] = d['Close'].rolling(STOCK_FAST_SMA).mean()
        d[f'SMA{STOCK_SLOW_SMA}'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()
        universe[ticker] = d
    return universe

required = set(recent['ticker'].unique())
print(f'Loading {len(required)} tickers for SMA check...')
universe_recent = load_required_tickers(DATA_DIR, required)
print(f'Loaded {len(universe_recent)} tickers.')

def stock_uptrend_on(ticker, date):
    df = universe_recent.get(ticker)
    if df is None: return None
    row = df[df['Date'] == date]
    if len(row) == 0: return None
    s50  = row[f'SMA{STOCK_FAST_SMA}'].iloc[0]
    s200 = row[f'SMA{STOCK_SLOW_SMA}'].iloc[0]
    if pd.isna(s50) or pd.isna(s200): return None
    return bool(s50 > s200)

recent['stock_uptrend_on_entry'] = recent.apply(
    lambda r: stock_uptrend_on(r['ticker'], r['entry_date']), axis=1
)
# Latest close for context
def latest_close(ticker):
    df = universe_recent.get(ticker)
    if df is None or len(df)==0: return None
    return df['Close'].iloc[-1]
recent['latest_close'] = recent['ticker'].apply(latest_close)
recent['pct_to_target']  = (recent['target_price'] - recent['latest_close']) / recent['latest_close']
recent['pct_to_stop']    = (recent['stop_price']   - recent['latest_close']) / recent['latest_close']
recent['status'] = np.where(
    recent['latest_close'] >= recent['target_price'], 'target_hit',
    np.where(recent['latest_close'] <= recent['stop_price'], 'stopped_out',
             'open'))

deployed = recent[recent['stock_uptrend_on_entry'] == True].copy()
print(f'\nRecent breakouts passing v2b filter (stock 50>200): {len(deployed)} of {len(recent)}')

if len(deployed):
    print('\nDEPLOYED v2b -- ACTIVE/RECENT TRADES:')
    print('='*100)
    show_cols = ['ticker','entry_date','days_since_entry','latest_close','entry_price',
                 'stop_price','target_price','pct_to_target','pct_to_stop','status',
                 'cup_depth_pct','handle_retr_pct']
    print(deployed[show_cols].to_string(index=False, float_format=lambda x: f'{x:.2f}'))


In [ ]:
# Cell 4 -- Phase 2: Pending setup detector (no breakout required)
def compute_atr(df, period=14):
    h,l,c = df['High'],df['Low'],df['Close']; pc = c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return tr.rolling(period).mean()

def find_pivot_highs(df, left=5, right=5):
    h = df['High'].values; n=len(h); piv=[]
    for i in range(left, n-right):
        w = h[i-left:i+right+1]
        if h[i]==w.max() and (w==h[i]).sum()==1: piv.append(i)
    return piv

def find_pivot_lows(df, left=5, right=5):
    l = df['Low'].values; n=len(l); piv=[]
    for i in range(left, n-right):
        w = l[i-left:i+right+1]
        if l[i]==w.min() and (w==l[i]).sum()==1: piv.append(i)
    return piv

def detect_pending_cup_handle(df, ticker,
                              pivot_left=5, pivot_right=5,
                              min_cup_duration=30,
                              max_cup_duration=130,
                              min_cup_depth_pct=0.12,
                              max_cup_depth_pct=0.50,
                              rch_lch_tolerance=0.15,
                              min_rounded_bars=3,
                              rounded_band_pct=0.05,
                              rounded_span_min=5,
                              max_handle_duration=30,
                              min_handle_retr_pct=0.10,
                              max_handle_retr_pct=0.50,
                              handle_max_age_bars=30,    # NEW: handle pivot low within last N bars
                              lookback_bars=220):         # NEW: scan only last N bars for speed
    """
    Returns pending setups: cup + handle formed but breakout has NOT yet confirmed,
    OR breakout confirmed within last 2 bars (just-fired).
    """
    df = df.copy()
    df['ATR'] = compute_atr(df, ATR_PERIOD)
    n = len(df)
    if n < lookback_bars + 50: return []
    # Restrict to recent window (with extra room for SMAs)
    start = max(0, n - lookback_bars)
    work = df.iloc[start:].reset_index(drop=True)
    offset = start

    pivot_highs = find_pivot_highs(work, pivot_left, pivot_right)
    pivot_lows  = find_pivot_lows(work, pivot_left, pivot_right)
    if len(pivot_highs) < 2: return []
    fires = []
    w_n = len(work)

    for i_lch, lch in enumerate(pivot_highs[:-1]):
        lch_high = work['High'].iloc[lch]
        for rch in pivot_highs[i_lch+1:]:
            dur = rch - lch
            if dur < min_cup_duration: continue
            if dur > max_cup_duration: break
            rch_high = work['High'].iloc[rch]
            if abs(rch_high - lch_high) / lch_high > rch_lch_tolerance: continue
            cup_slice_lows = work['Low'].iloc[lch+1:rch].values
            if len(cup_slice_lows) < 5: continue
            cup_low = cup_slice_lows.min()
            cup_low_idx = lch + 1 + int(cup_slice_lows.argmin())
            cup_rim = (lch_high + rch_high) / 2
            cup_depth_pct = (cup_rim - cup_low) / cup_rim
            if cup_depth_pct < min_cup_depth_pct: continue
            if cup_depth_pct > max_cup_depth_pct: continue
            band_threshold = cup_low * (1 + rounded_band_pct)
            cup_full_lows = work['Low'].iloc[lch:rch+1].values
            in_band = [k for k,v in enumerate(cup_full_lows) if v <= band_threshold]
            if len(in_band) < min_rounded_bars: continue
            if (max(in_band) - min(in_band)) < rounded_span_min: continue

            # Handle: pivot low within max_handle_duration after RCH, AND within handle_max_age_bars
            handle_candidates = [p for p in pivot_lows
                                  if rch < p <= rch + max_handle_duration
                                  and p >= w_n - handle_max_age_bars]
            if not handle_candidates: continue
            handle_low_idx = min(handle_candidates, key=lambda p: work['Low'].iloc[p])
            handle_low = work['Low'].iloc[handle_low_idx]
            handle_drop = rch_high - handle_low
            cup_depth_abs = cup_rim - cup_low
            if cup_depth_abs <= 0: continue
            retr = handle_drop / cup_depth_abs
            if retr < min_handle_retr_pct: continue
            if retr > max_handle_retr_pct: continue
            if handle_low <= cup_low: continue

            breakout_level = max(lch_high, rch_high)
            # Has breakout already happened? Check after handle_low
            confirm_idx = None
            for j in range(handle_low_idx+1, w_n):
                if work['Close'].iloc[j] > breakout_level:
                    confirm_idx = j; break

            atr = work['ATR'].iloc[rch]
            if pd.isna(atr) or atr<=0: continue
            current_price = work['Close'].iloc[-1]

            # Status
            if confirm_idx is None:
                status = 'pending'
                entry_price_est = breakout_level * 1.001
                bars_since_handle = w_n - 1 - handle_low_idx
            elif (w_n - 1) - confirm_idx <= 2:
                status = 'just_fired'
                entry_price_est = work['Open'].iloc[min(confirm_idx+1, w_n-1)]
                bars_since_handle = w_n - 1 - handle_low_idx
            else:
                continue  # Breakout too old for this scan; cached fires already cover it

            raw_stop = handle_low * 0.999
            stop_dist = entry_price_est - raw_stop
            if stop_dist <= 0: continue
            if stop_dist < MIN_STOP_ATR*atr: stop_dist = MIN_STOP_ATR*atr
            if stop_dist/entry_price_est > MAX_STOP_PCT: continue
            stop_price = entry_price_est - stop_dist
            target_price = breakout_level + cup_depth_abs

            fires.append({
                'ticker': ticker, 'status': status,
                'lch_date':        work['Date'].iloc[lch].date(),
                'cup_low_date':    work['Date'].iloc[cup_low_idx].date(),
                'rch_date':        work['Date'].iloc[rch].date(),
                'handle_low_date': work['Date'].iloc[handle_low_idx].date(),
                'bars_since_handle': bars_since_handle,
                'current_price':   round(current_price,2),
                'breakout_level':  round(breakout_level,2),
                'pct_to_breakout': round((breakout_level - current_price)/current_price * 100, 2),
                'entry_est':       round(entry_price_est,2),
                'stop':            round(stop_price,2),
                'target':          round(target_price,2),
                'cup_depth_pct':   round(cup_depth_pct*100,1),
                'handle_retr_pct': round(retr*100,1),
                'cup_duration':    dur,
                'handle_dur_days':  (work['Date'].iloc[handle_low_idx] - work['Date'].iloc[rch]).days,
                'handle_depth_atr': round((breakout_level - handle_low) / atr, 3) if atr else None,
                'atr':             round(atr,2),
            })
            break  # one cup per LCH

    if not fires: return []
    out = pd.DataFrame(fires).sort_values('handle_low_date', ascending=False)
    out = out.drop_duplicates(['ticker','handle_low_date'], keep='first')
    return out.to_dict('records')

print('Pending detector defined.')


In [ ]:
# Cell 5 -- Run pending scan on the FULL universe (10-15 min)
import time

t0 = time.time()
csv_files = sorted(glob.glob(f'{DATA_DIR}/*.csv'))
csv_files = [f for f in csv_files if Path(f).stem.upper() != 'SPY']
print(f'Scanning {len(csv_files)} tickers for pending setups...')

all_pending = []
loaded = 0
for f in csv_files:
    ticker = Path(f).stem.upper()
    try:
        d = pd.read_csv(f, usecols=['Date','Open','High','Low','Close'])
        d['Date'] = pd.to_datetime(d['Date'])
        d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
        if len(d) < 250: continue
        d[f'SMA{STOCK_FAST_SMA}'] = d['Close'].rolling(STOCK_FAST_SMA).mean()
        d[f'SMA{STOCK_SLOW_SMA}'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()

        # Cheap pre-filter: stock currently in uptrend (50 > 200)?
        s50_now  = d[f'SMA{STOCK_FAST_SMA}'].iloc[-1]
        s200_now = d[f'SMA{STOCK_SLOW_SMA}'].iloc[-1]
        if pd.isna(s50_now) or pd.isna(s200_now) or s50_now <= s200_now:
            continue

        results = detect_pending_cup_handle(d, ticker)
        if results:
            for r in results:
                r['stock_50_gt_200'] = True
            all_pending.extend(results)
        loaded += 1
        if loaded % 100 == 0:
            print(f'  {loaded} tickers scanned, {len(all_pending)} setups found ({time.time()-t0:.0f}s)')
    except Exception as e:
        pass

pending_df = pd.DataFrame(all_pending) if all_pending else pd.DataFrame()
print(f'\nDone in {time.time()-t0:.0f}s.')
print(f'Total pending setups: {len(pending_df)}')
if len(pending_df):
    print(f'  pending breakout: {(pending_df["status"]=="pending").sum()}')
    print(f'  just fired (<=2 days): {(pending_df["status"]=="just_fired").sum()}')


In [ ]:
# Cell 6 -- Report and save
if len(pending_df) == 0:
    print('No pending setups found.')
else:
    pending_df = pending_df.sort_values(['status','bars_since_handle'])

    print('\n' + '='*100)
    print('  PENDING BREAKOUTS (handle formed, awaiting close above breakout level)')
    print('='*100)
    pending = pending_df[pending_df['status']=='pending'].copy()
    if len(pending):
        cols = ['ticker','handle_low_date','bars_since_handle','current_price','breakout_level',
                'pct_to_breakout','stop','target','cup_depth_pct','handle_retr_pct','cup_duration']
        print(pending[cols].to_string(index=False))
    else:
        print('  (none)')

    print('\n' + '='*100)
    print('  JUST FIRED (broke out in last 2 days)')
    print('='*100)
    fired = pending_df[pending_df['status']=='just_fired'].copy()
    if len(fired):
        cols = ['ticker','handle_low_date','current_price','entry_est','stop','target',
                'cup_depth_pct','handle_retr_pct','cup_duration']
        print(fired[cols].to_string(index=False))
    else:
        print('  (none)')

    # Save
    pending_df.to_csv(f'{OUT_DIR}/cup_handle_active_setups.csv', index=False)
    print(f'\nSaved to {OUT_DIR}/cup_handle_active_setups.csv')


## Output

Two tables to paste back:
1. Cell 3 -- recent breakouts (last 30 days), with current status (open/target_hit/stopped)
2. Cell 6 -- pending setups (in handle now) and just-fired (broke out last 2 days)

**For charting:**
- LCH date = left rim
- Cup low date = bottom of cup
- RCH date = right rim
- Handle low date = current handle pivot
- Breakout level = horizontal resistance line
- Stop = below handle low
- Target = breakout level + cup depth

The `pct_to_breakout` column on pending setups tells you how far the stock has to rally to trigger the trade.
